In [ ]:
%pip install --upgrade openai pydantic mermaid-py

# SQL + Data Analysis Agent

Build an AI agent that turns plain-English questions into SQL queries, runs them safely against a database, and explains the results back to you.

## What You Will Build

In this notebook you will create a complete **text-to-SQL agent**. You give it a question like *"Who are our top customers?"* and it will:

1. **Inspect the database schema** so it knows what tables and columns exist.
2. **Generate a SQL query** that answers your question.
3. **Validate the query** to make sure it is read-only (no accidental deletes!).
4. **Execute the query** and return the raw results.
5. **Interpret the data** and give you a clear, plain-English answer with insights.

Here is the flow at a glance:

Why does the agent inspect the schema *before* writing SQL? Because giving the model the real table and column names dramatically reduces errors like referencing columns that don't exist.

In [ ]:
import mermaid as md

md.Mermaid("""
sequenceDiagram
    participant User
    participant Agent as Agent (LLM)
    participant DB as SQLite Database

    User->>Agent: Top customers?
    Agent->>DB: inspect_schema()
    DB-->>Agent: schema details
    Agent->>DB: validate + execute SELECT
    DB-->>Agent: query results
    Agent-->>User: Plain-English answer + insights
""")

## Safety First

Text-to-SQL agents talk directly to databases, which means **safety guardrails are essential**. Without them, a model could accidentally (or through prompt injection) run destructive queries.

Our agent implements several layers of protection:

| Guardrail | Purpose |
|-----------|---------|
| **SQL validation** | Reject any `DROP`, `DELETE`, `UPDATE`, `INSERT`, `ALTER`, `CREATE`, or `TRUNCATE` statements before they reach the database |
| **Row limits** | Automatically append `LIMIT` clauses to prevent queries that return millions of rows and exhaust memory |
| **Read-only access** | The agent is explicitly designed for `SELECT` queries only |
| **Schema inspection** | Real schema access reduces hallucinated table/column names that would cause errors |

In a production environment you would also want:
- A read-only database user/connection
- Query execution timeouts
- Query complexity limits (e.g., no more than N joins)
- Logging and auditing of all executed queries

## Setup

Let's get the imports and configuration out of the way. We will use `gpt-5-mini` as our model and cap query results at 50 rows for safety.

In [ ]:
import os
import json
import sqlite3
import pathlib
from openai import OpenAI
from pydantic import BaseModel, ConfigDict, Field

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")

client = OpenAI()
MODEL = "gpt-5-mini"
MAX_ROW_LIMIT = 50  # Safety: cap the number of rows any query can return

## Load Data from Resources

Instead of hard-coding data inside the notebook, we load everything from external resource files. The JSON file contains our sample e-commerce data (customers, products, orders, and order items), and the text file holds the system prompt that tells the agent how to behave.

In [ ]:
RESOURCES = pathlib.Path("resources")

# Load sample data
data = json.loads((RESOURCES / "data" / "sql_analysis_sample.json").read_text())

# Load the agent's system instructions
instructions = (RESOURCES / "prompts" / "sql_data_agent_instructions.txt").read_text()

print(f"Loaded {len(data['customers'])} customers, {len(data['products'])} products, "
      f"{len(data['orders'])} orders, {len(data['order_items'])} order items")
print(f"\nSystem instructions ({len(instructions)} chars):")
print(instructions[:200] + "...")

## Create the Database

Now let's spin up an in-memory SQLite database and populate it from the JSON data we just loaded. The schema represents a small e-commerce company with four tables:

- **customers**: Who is buying (name, location, lifetime spend)
- **products**: What is for sale (name, category, price, stock)
- **orders**: Each purchase transaction
- **order_items**: Individual line items within each order

The sample data is designed to produce interesting analytical results, with varied countries, product categories, date ranges, and spending patterns.

In [ ]:
# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")

# Create tables
conn.executescript("""
CREATE TABLE customers (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT NOT NULL,
    city TEXT NOT NULL,
    country TEXT NOT NULL,
    signup_date TEXT NOT NULL,
    lifetime_spend REAL NOT NULL
);

CREATE TABLE products (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL,
    stock_quantity INTEGER NOT NULL
);

CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    order_date TEXT NOT NULL,
    total_amount REAL NOT NULL,
    status TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(id)
);

CREATE TABLE order_items (
    id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    product_id INTEGER NOT NULL,
    quantity INTEGER NOT NULL,
    unit_price REAL NOT NULL,
    FOREIGN KEY (order_id) REFERENCES orders(id),
    FOREIGN KEY (product_id) REFERENCES products(id)
);
""")

# Populate from JSON
for c in data["customers"]:
    conn.execute(
        "INSERT INTO customers VALUES (?,?,?,?,?,?,?)",
        (c["id"], c["name"], c["email"], c["city"], c["country"], c["signup_date"], c["lifetime_spend"]),
    )

for p in data["products"]:
    conn.execute(
        "INSERT INTO products VALUES (?,?,?,?,?)",
        (p["id"], p["name"], p["category"], p["price"], p["stock_quantity"]),
    )

for o in data["orders"]:
    conn.execute(
        "INSERT INTO orders VALUES (?,?,?,?,?)",
        (o["id"], o["customer_id"], o["order_date"], o["total_amount"], o["status"]),
    )

for oi in data["order_items"]:
    conn.execute(
        "INSERT INTO order_items VALUES (?,?,?,?,?)",
        (oi["id"], oi["order_id"], oi["product_id"], oi["quantity"], oi["unit_price"]),
    )

conn.commit()
print("Database created successfully!")
print(f"  - {conn.execute('SELECT COUNT(*) FROM customers').fetchone()[0]} customers")
print(f"  - {conn.execute('SELECT COUNT(*) FROM products').fetchone()[0]} products")
print(f"  - {conn.execute('SELECT COUNT(*) FROM orders').fetchone()[0]} orders")
print(f"  - {conn.execute('SELECT COUNT(*) FROM order_items').fetchone()[0]} order items")

## Define the Tools with Pydantic

Here is where things get interesting. Instead of hand-typing JSON schemas for our tool definitions (which is tedious and error-prone), we will use **Pydantic models** to generate them automatically.

The pattern works like this:

1. Create a `ToolArgs` base class that forbids extra fields (required for OpenAI's strict mode).
2. Define a small model for each tool's parameters.
3. Use a `function_tool()` helper to combine the name, description, and Pydantic-generated schema into the format OpenAI expects.

Why is this better? You get automatic validation, clear documentation, and you never have to worry about mistyped JSON schemas again.

In [ ]:
class ToolArgs(BaseModel):
    """Base for tool argument models. Forbids extra fields for strict mode."""
    model_config = ConfigDict(extra="forbid")


class InspectSchemaArgs(ToolArgs):
    """No parameters needed; the agent just calls this to see the schema."""
    pass


class ExecuteSQLArgs(ToolArgs):
    """Parameters for the SQL execution tool."""
    query: str = Field(..., description="The SQL SELECT query to execute.")


def function_tool(name: str, description: str, args_model: type[BaseModel]) -> dict:
    """Build an OpenAI function-tool dict from a Pydantic model."""
    return {
        "type": "function",
        "name": name,
        "description": description,
        "parameters": args_model.model_json_schema(),
        "strict": True,
    }

## Implement Tool Functions

Now let's write the actual Python functions that power each tool.

`inspect_schema` returns the full database structure: CREATE TABLE statements, a few sample rows from each table, and row counts. The agent calls this first so it knows exactly what data is available.

`validate_and_execute_sql` is the workhorse. It checks the query for forbidden keywords, appends a LIMIT clause if needed, runs the query, and formats the results as a readable table.

In [ ]:
def inspect_schema():
    """Return the full database schema with CREATE TABLE statements and sample data."""
    schema_parts = []

    # Get all table names
    tables = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()

    for (table_name,) in tables:
        # Get the CREATE TABLE statement
        create_stmt = conn.execute(
            "SELECT sql FROM sqlite_master WHERE type='table' AND name=?",
            (table_name,),
        ).fetchone()[0]
        schema_parts.append(create_stmt + ";")

        # Get sample rows (first 3)
        cursor = conn.execute(f"SELECT * FROM {table_name} LIMIT 3")
        columns = [desc[0] for desc in cursor.description]
        rows = cursor.fetchall()

        schema_parts.append(f"\n-- Sample data from {table_name}:")
        schema_parts.append("-- " + " | ".join(columns))
        for row in rows:
            schema_parts.append("-- " + " | ".join(str(v) for v in row))
        schema_parts.append("")

    # Get row counts
    schema_parts.append("-- Row counts:")
    for (table_name,) in tables:
        count = conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        schema_parts.append(f"-- {table_name}: {count} rows")

    return "\n".join(schema_parts)


def validate_and_execute_sql(query):
    """Validate and execute a SQL query with safety checks."""
    # Safety: reject any write operations
    forbidden = ["INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "CREATE", "TRUNCATE"]
    query_upper = query.upper().strip()
    for keyword in forbidden:
        if keyword in query_upper.split():
            return f"Error: {keyword} operations are not allowed. This agent is read-only."

    # Add LIMIT if not present to prevent huge result sets
    if "LIMIT" not in query_upper:
        query = query.rstrip(";") + f" LIMIT {MAX_ROW_LIMIT}"

    try:
        cursor = conn.execute(query)
        columns = [desc[0] for desc in cursor.description]
        rows = cursor.fetchall()

        # Format results as a readable table
        result = " | ".join(columns) + "\n"
        result += "-" * len(result) + "\n"
        for row in rows:
            result += " | ".join(str(v) for v in row) + "\n"
        result += f"\n({len(rows)} rows returned)"
        return result
    except Exception as e:
        return f"SQL Error: {str(e)}"


# Quick test: verify the tools work
print("=== Schema Preview (first 20 lines) ===")
print("\n".join(inspect_schema().split("\n")[:20]))
print("\n...\n")
print("=== Sample Query ===")
print(validate_and_execute_sql(
    "SELECT name, country, lifetime_spend FROM customers ORDER BY lifetime_spend DESC LIMIT 5"
))

## Generate Tool Schemas

Let's build the tool definitions by calling `function_tool()` with our Pydantic models. Notice how the JSON schema is generated automatically from the model fields. No more hand-typing `"type": "object", "properties": {...}`!

Take a look at the printed output to see exactly what Pydantic produces for us.

In [ ]:
tools = [
    function_tool(
        "inspect_schema",
        (
            "Inspect the database schema. Returns all CREATE TABLE statements, "
            "sample data from each table, and row counts. Call this FIRST before "
            "writing any SQL queries so you know what tables and columns exist."
        ),
        InspectSchemaArgs,
    ),
    function_tool(
        "validate_and_execute_sql",
        (
            "Validate and execute a SQL query against the database. "
            "The query must be a SELECT statement; write operations (INSERT, UPDATE, "
            "DELETE, DROP, etc.) are not allowed. A row LIMIT is automatically applied "
            "if not present. Returns the results as a formatted table."
        ),
        ExecuteSQLArgs,
    ),
]

print("Tools defined:")
for tool in tools:
    print(f"  - {tool['name']}")
print()
print("Here is what Pydantic generated for validate_and_execute_sql:")
print(json.dumps(tools[1], indent=2))

## The Agent Loop

This is the heart of the system. Here is how it works:

1. Your question is sent to the model along with the tool definitions.
2. If the model responds with tool calls, we execute them and feed the results back.
3. This continues until the model produces a final text response (no more tool calls).
4. We cap the loop at 5 iterations as a safety measure.

The system instructions (loaded from our resource file) tell the model to always inspect the schema first, write clean SQL, and provide clear explanations.

In [ ]:
def ask_data_question(question):
    """
    Agent loop: takes a natural language question, uses tools to inspect
    schema and run SQL, then provides a plain-English answer.
    """
    messages = [{"role": "user", "content": question}]

    print(f"Question: {question}")
    print("=" * 60)

    for turn in range(5):  # Max 5 tool-calling turns
        response = client.responses.create(
            model=MODEL,
            instructions=instructions,
            input=messages,
            tools=tools,
        )

        # Check for tool calls in the response
        tool_calls = [item for item in response.output if item.type == "function_call"]

        # If no tool calls, the model produced a final answer
        if not tool_calls:
            answer = response.output_text
            print(f"\nAnswer:\n{answer}")
            return answer

        # Process each tool call
        for tool_call in tool_calls:
            args = json.loads(tool_call.arguments)

            if tool_call.name == "inspect_schema":
                print(f"\n[Turn {turn + 1}] Inspecting database schema...")
                result = inspect_schema()
            elif tool_call.name == "validate_and_execute_sql":
                print(f"\n[Turn {turn + 1}] Executing SQL:")
                print(f"  {args['query']}")
                result = validate_and_execute_sql(args["query"])
                print(f"\n[Results]\n{result}")
            else:
                result = f"Unknown tool: {tool_call.name}"

            # Append the tool call and its output to the conversation
            messages.append(tool_call)
            messages.append({
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": result,
            })

    return "Max turns reached without a final answer."


print("Agent loop defined. Ready to answer questions!")

## Try It Out

Time to put the agent through its paces! We will ask six different types of analytical questions, from simple aggregations to subqueries. Watch how the agent inspects the schema, writes SQL, and interprets the results for each one.

### Question 1: Aggregation

Let's start simple. Who are the biggest spenders? This requires summing order totals and ranking customers.

In [ ]:
ask_data_question("What are the top 5 customers by total spending?")

### Question 2: Joins + Aggregation

Now let's make the agent work a little harder. Which product category brings in the most revenue? This requires joining `order_items` with `products` and grouping by category.

In [ ]:
ask_data_question("Which product category generates the most revenue?")

### Question 3: Date Analysis

How about trends over time? The agent needs to extract the month from `order_date` and count orders per month.

In [ ]:
ask_data_question("How many orders were placed per month in 2024?")

### Question 4: Grouping by Country

What does spending look like across different countries? This requires joining customers with orders and computing averages per country.

In [ ]:
ask_data_question("What is the average order value by country?")

### Question 5: LEFT JOIN + NULL Check

Here is a classic pattern: finding products that have *never* been ordered. The agent needs a LEFT JOIN and a NULL check on `order_items`.

In [ ]:
ask_data_question("Which products have never been ordered?")

### Question 6: Subquery

Last one. Can the agent figure out which customers spend more than average? This requires computing the average first, then filtering customers above it.

In [ ]:
ask_data_question("Show me customers who spent more than the average")

## Safety Demo

Let's make sure our guardrails actually work. What happens if someone asks the agent to do something destructive? And what happens if we try to sneak forbidden SQL past the validation function directly?

In [ ]:
# Ask the agent to delete data (it should refuse)
ask_data_question("Delete all customers from the database")

In [ ]:
# Test the safety check directly to confirm it works
print("Direct safety check tests:")
print()
print("1. DELETE:", validate_and_execute_sql("DELETE FROM customers WHERE id = 1"))
print()
print("2. DROP:", validate_and_execute_sql("DROP TABLE customers"))
print()
print("3. UPDATE:", validate_and_execute_sql("UPDATE customers SET name = 'hacked' WHERE id = 1"))
print()
print("4. Valid SELECT (should work):")
print(validate_and_execute_sql("SELECT COUNT(*) as total_customers FROM customers"))

## Your Turn: Build a Library Catalog Agent

Now for the real challenge! You will build a complete text-to-SQL agent from scratch for a library database. Same pattern you just learned, but this time you write all the pieces yourself.

## Exercise 2: Build a Library Catalog Agent

Now for the real challenge! You will build a **text-to-SQL agent from scratch** for a library database. This is the same pattern you saw in the SQL + Data Analysis Agent notebook, but this time you are writing all the pieces yourself.

The database has three tables:

| Table | Description |
|-------|-------------|
| `books` | The library's catalog (title, author, genre, year, availability) |
| `members` | Library members (name, email, membership type) |
| `loans` | Which books are checked out by which members |

Your agent will have two tools:

| Tool | Purpose |
|------|---------|
| `inspect_library_schema` | Show the database schema so the model knows what tables and columns exist |
| `execute_library_query` | Validate and execute a read-only SQL query against the library database |

### Provided Code: Library Database

The cell below creates and populates the library database. **Do not modify it.** Read through the schema and sample data so you understand what is available.

In [ ]:
# ── Create the library database ─────────────────────────────────────────────
library_conn = sqlite3.connect(":memory:")

library_conn.executescript("""
CREATE TABLE books (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    genre TEXT NOT NULL,
    year_published INTEGER NOT NULL,
    available INTEGER NOT NULL DEFAULT 1
);

CREATE TABLE members (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT NOT NULL,
    membership_type TEXT NOT NULL CHECK(membership_type IN ('basic', 'premium', 'student'))
);

CREATE TABLE loans (
    id INTEGER PRIMARY KEY,
    book_id INTEGER NOT NULL,
    member_id INTEGER NOT NULL,
    loan_date TEXT NOT NULL,
    return_date TEXT,
    status TEXT NOT NULL CHECK(status IN ('active', 'returned', 'overdue')),
    FOREIGN KEY (book_id) REFERENCES books(id),
    FOREIGN KEY (member_id) REFERENCES members(id)
);
""")

# ── Populate books ───────────────────────────────────────────────────────────
books = [
    (1, "The Great Gatsby", "F. Scott Fitzgerald", "Fiction", 1925, 1),
    (2, "To Kill a Mockingbird", "Harper Lee", "Fiction", 1960, 0),
    (3, "1984", "George Orwell", "Dystopian", 1949, 1),
    (4, "Pride and Prejudice", "Jane Austen", "Romance", 1813, 1),
    (5, "The Catcher in the Rye", "J.D. Salinger", "Fiction", 1951, 0),
    (6, "Brave New World", "Aldous Huxley", "Dystopian", 1932, 1),
    (7, "The Hobbit", "J.R.R. Tolkien", "Fantasy", 1937, 0),
    (8, "Fahrenheit 451", "Ray Bradbury", "Dystopian", 1953, 1),
    (9, "Jane Eyre", "Charlotte Bronte", "Romance", 1847, 1),
    (10, "Dune", "Frank Herbert", "Science Fiction", 1965, 1),
]
library_conn.executemany("INSERT INTO books VALUES (?,?,?,?,?,?)", books)

# ── Populate members ─────────────────────────────────────────────────────────
members = [
    (1, "Alice Johnson", "alice@library.org", "premium"),
    (2, "Bob Smith", "bob@library.org", "basic"),
    (3, "Carol Davis", "carol@library.org", "student"),
    (4, "David Wilson", "david@library.org", "premium"),
    (5, "Eve Martinez", "eve@library.org", "student"),
]
library_conn.executemany("INSERT INTO members VALUES (?,?,?,?)", members)

# ── Populate loans ───────────────────────────────────────────────────────────
loans = [
    (1, 2, 1, "2025-01-10", None, "active"),
    (2, 5, 3, "2025-01-15", None, "active"),
    (3, 7, 2, "2024-12-20", None, "overdue"),
    (4, 1, 4, "2024-11-05", "2024-12-01", "returned"),
    (5, 3, 5, "2024-10-15", "2024-11-10", "returned"),
    (6, 4, 1, "2025-01-20", None, "active"),
    (7, 10, 3, "2024-12-01", "2024-12-28", "returned"),
    (8, 6, 2, "2025-01-25", None, "active"),
]
library_conn.executemany("INSERT INTO loans VALUES (?,?,?,?,?,?)", loans)

library_conn.commit()

print("Library database created!")
print(f"  books:   {library_conn.execute('SELECT COUNT(*) FROM books').fetchone()[0]} rows")
print(f"  members: {library_conn.execute('SELECT COUNT(*) FROM members').fetchone()[0]} rows")
print(f"  loans:   {library_conn.execute('SELECT COUNT(*) FROM loans').fetchone()[0]} rows")

### Your Code: Define Pydantic Models

Create two Pydantic argument models:

1. **`InspectLibrarySchemaArgs`**: Takes no parameters. The agent calls this to see the database schema.
2. **`ExecuteLibraryQueryArgs`**: Takes a single required field `query` (str) for the SQL SELECT statement to execute.

In [ ]:
# YOUR CODE HERE: Define InspectLibrarySchemaArgs
class InspectLibrarySchemaArgs(ToolArgs):
    pass  # This tool takes no parameters (that's correct!)


# YOUR CODE HERE: Define ExecuteLibraryQueryArgs
class ExecuteLibraryQueryArgs(ToolArgs):
    pass  # Replace with your field(s)

### Your Code: Implement Tool Functions

Write the two tool functions:

1. **`inspect_library_schema()`**: Return the CREATE TABLE statements, 3 sample rows from each table, and row counts. (Hint: query `sqlite_master` for the schema, then run `SELECT * FROM <table> LIMIT 3` for samples.)
2. **`execute_library_query(query)`**: Validate that the query is read-only (reject `DROP`, `DELETE`, `UPDATE`, `INSERT`, `ALTER`, `CREATE`, `TRUNCATE`), add a LIMIT if missing, execute it, and return formatted results. Return a clear error message if something goes wrong.

In [ ]:
MAX_ROWS = 50


# YOUR CODE HERE: Implement inspect_library_schema
def inspect_library_schema():
    """Return the library database schema with CREATE TABLE statements and sample data."""
    pass  # Your implementation here


# YOUR CODE HERE: Implement execute_library_query
def execute_library_query(query):
    """Validate and execute a read-only SQL query against the library database."""
    pass  # Your implementation here

In [ ]:
# Quick sanity check (uncomment after implementing your functions)
# print("=== Schema ===")
# print(inspect_library_schema()[:500])
# print()
# print("=== Sample query ===")
# print(execute_library_query("SELECT title, author FROM books WHERE available = 1"))

### Your Code: Register Tools and Build the Agent Loop

Now assemble everything:

1. Create the `library_tools` list using `function_tool()`.
2. Write the `ask_library_question()` agent loop that sends the user's question to the model, handles tool calls, and returns a final text answer.

**Hint:** Use the `ask_data_question` function from the SQL notebook as a reference. The pattern is the same: call the model, check for tool calls, dispatch them, feed results back, repeat.

In [ ]:
# System instructions for the library agent
library_instructions = """You are a helpful library catalog assistant. You help patrons
find books, check availability, and look up loan information.

ALWAYS inspect the database schema first before writing any SQL queries.
Only use SELECT statements. Never modify data.
When presenting results, be friendly and format the information clearly."""


# YOUR CODE HERE: Build the library_tools list
library_tools = [
    # Add your two tools here using function_tool()
]


# YOUR CODE HERE: Write the agent loop
def ask_library_question(question, max_turns=5, verbose=True):
    """Agent loop for the library catalog assistant."""
    pass  # Your implementation here

### Test Your Library Agent

Run the test cells below to see if your agent can answer questions about the library database. If everything is wired up correctly, the agent will inspect the schema, write SQL, and explain the results.

In [ ]:
# Test 1: Which books are currently checked out?
answer = ask_library_question("Which books are currently checked out and who has them?")
print(f"\nFINAL ANSWER:\n{answer}")

In [ ]:
# Test 2: Available books in a specific genre
answer = ask_library_question("What dystopian books are available to borrow right now?")
print(f"\nFINAL ANSWER:\n{answer}")

In [ ]:
# Test 3: Member activity
answer = ask_library_question("Which member has borrowed the most books?")
print(f"\nFINAL ANSWER:\n{answer}")

In [ ]:
# Test 4: Overdue books
answer = ask_library_question("Are there any overdue books? If so, who has them and when were they due?")
print(f"\nFINAL ANSWER:\n{answer}")

## Summary

Great work! You just built a complete SQL Data Analysis Agent. Here are the key takeaways:

- **Text-to-SQL agents** convert natural language into database queries, making data accessible to non-technical users.
- **Pydantic models** generate tool schemas automatically, eliminating hand-typed JSON and reducing errors.
- **Schema inspection** is a critical first step. Giving the model access to the real schema dramatically reduces hallucinated table and column names.
- **SQL validation is essential.** Always reject write operations (`DROP`, `DELETE`, `UPDATE`, etc.) before they reach the database.
- **Row limits** prevent memory issues and protect against queries that return enormous result sets.
- **The model can interpret results** and provide business insights, going beyond raw numbers to explain what the data means.
- **This pattern generalizes** to any SQL database (PostgreSQL, MySQL, etc.). Just swap the connection and adjust the schema inspection logic.